# 🗺️ Contact Map Fingerprinting: Classifying Protein Folds from Structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/synth-pdb/blob/master/examples/ml_integration/contact_map_fingerprinting.ipynb)

**Duration:** ~30 minutes | **Level:** ⭐⭐⭐ Advanced

---

## The core idea

A **contact map** is a 2D matrix where entry (i, j) is the Cα distance between
residues i and j. It encodes the entire 3D topology of a protein into a single
fixed-size image — all without coordinates, chain IDs, or atom names.

Different fold classes leave strikingly different visual signatures:

| Fold | Contact map pattern | Why |
| :--- | :--- | :--- |
| **Alpha helix** | Diagonal bands at \|i-j\| = 3, 4 | Residues 3–4 apart are hydrogen-bonded |
| **Beta strand** | Parallel or anti-parallel off-diagonal bands | Extended backbone, sheet hydrogen bonds |
| **Random coil** | Sparse, noisy — only nearest neighbours | No persistent secondary structure |

In this tutorial we:
1. Use `BatchedGenerator` to produce thousands of labelled structures in seconds
2. Compute contact map fingerprints with `compute_contact_map()`
3. Visualise the fold-specific patterns
4. Train a logistic-regression classifier that distinguishes the three fold classes
5. Inspect which contact-pairs are most informative — connecting ML weights back to biology


## 🔧 Setup


In [ ]:
import os, sys, io, time

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Google Colab — installing dependencies...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'synth-pdb', 'biotite'], check=True)
else:
    repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f'Running locally: {repo_root}')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biotite.structure.io.pdb as pdb_io
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from synth_pdb.batch_generator import BatchedGenerator
from synth_pdb.contact import compute_contact_map

print('All imports successful.')


## ⚡ Step 1: Adaptive Generation Budget

Rather than hardcoding a dataset size, we benchmark the actual throughput
of the full pipeline on **this hardware** and scale to a time budget.

Faster processors or better hardware will automatically produce larger, richer
datasets — the tutorial improves without any code changes.

**To generate more structures:** increase `GENERATION_TIME_BUDGET_SECONDS` below.


In [ ]:
# ── Tuning knobs ──────────────────────────────────────────────────────────
# Increase this on faster hardware or when you have more time.
GENERATION_TIME_BUDGET_SECONDS = 30

# Structural parameters
N_RESIDUES   = 15   # contact map will be 15x15 = 225 features
THRESHOLD_ANG = 8.0  # Å — standard Cα contact distance cutoff
N_CLASSES    = 3    # alpha / beta / coil
MAX_PER_CLASS = 2000 # hard cap so training stays fast even on very fast hardware
MIN_PER_CLASS = 50   # statistical minimum for a meaningful classifier

# ── Benchmark: measure full generate→contact_map throughput ───────────────
print('Benchmarking pipeline throughput...')
SEQ = 'A' * N_RESIDUES
_gen  = BatchedGenerator(sequence_str=SEQ, n_batch=100)
_t0   = time.time()
_batch = _gen.generate_batch(conformation='alpha', seed=0)
for _i in range(100):
    _pdb = _batch.to_pdb(index=_i)
    _st  = pdb_io.PDBFile.read(io.StringIO(_pdb)).get_structure(model=1)
    compute_contact_map(_st, threshold=THRESHOLD_ANG)
_throughput = 100 / (time.time() - _t0)  # structures/sec

N_PER_CLASS = int(
    min(MAX_PER_CLASS,
        max(MIN_PER_CLASS,
            _throughput * GENERATION_TIME_BUDGET_SECONDS / N_CLASSES))
)

print(f'Throughput:       {_throughput:,.0f} structures/sec (full pipeline)')
print(f'Time budget:      {GENERATION_TIME_BUDGET_SECONDS}s')
print(f'Structures/class: {N_PER_CLASS}')
print(f'Total dataset:    {N_CLASSES * N_PER_CLASS} labelled structures')
print(f'Feature vector:   {N_RESIDUES}x{N_RESIDUES} = {N_RESIDUES**2} contact-pair distances')


## 🧬 Step 2: Generate a Labelled Dataset

`BatchedGenerator` produces `n_batch` structures in a single vectorised call.
We use three sequences:
- **Poly-Ala** in alpha conformation — strongest helix former, cleanest signal
- **Poly-Val** in beta conformation — Val strongly prefers sheet
- **Poly-Gly** with `drift=90` — Gly's lack of a sidechain allows full phi/psi
  freedom; adding drift scatters into random-coil territory


In [ ]:
def generate_class(sequence, conformation, n, drift=0.0, label=''):
    """Generate n contact-map fingerprints for one fold class."""
    gen   = BatchedGenerator(sequence_str=sequence, n_batch=n)
    batch = gen.generate_batch(conformation=conformation, drift=drift, seed=42)
    X = []
    for i in range(n):
        pdb_str   = batch.to_pdb(index=i)
        structure = pdb_io.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)
        cmap      = compute_contact_map(structure, threshold=THRESHOLD_ANG)
        X.append(cmap.flatten())
    return np.array(X, dtype=np.float32)

t_start = time.time()

print(f'Generating {N_PER_CLASS} alpha-helix structures...', end=' ')
X_alpha = generate_class('A' * N_RESIDUES, 'alpha', N_PER_CLASS)
print(f'done ({time.time()-t_start:.1f}s)')

t2 = time.time()
print(f'Generating {N_PER_CLASS} beta-strand structures...', end=' ')
X_beta  = generate_class('V' * N_RESIDUES, 'beta',  N_PER_CLASS)
print(f'done ({time.time()-t2:.1f}s)')

t3 = time.time()
print(f'Generating {N_PER_CLASS} random-coil structures...', end=' ')
X_coil  = generate_class('G' * N_RESIDUES, 'alpha', N_PER_CLASS, drift=90.0)
print(f'done ({time.time()-t3:.1f}s)')

# Build labelled arrays
X = np.vstack([X_alpha, X_beta, X_coil])
y = np.array(['alpha'] * N_PER_CLASS + ['beta'] * N_PER_CLASS + ['coil'] * N_PER_CLASS)

total_time = time.time() - t_start
print(f'\nDataset ready: {X.shape[0]} structures, {X.shape[1]} features each')
print(f'Total generation time: {total_time:.1f}s')


## 🎨 Step 3: Visualise the Fold-Specific Patterns

Each row shows three example contact maps from that fold class.
Note the diagonal stripe patterns — they encode the geometry of
secondary structure as clearly as a fingerprint:


In [ ]:
fig = plt.figure(figsize=(11, 7))
gs  = gridspec.GridSpec(3, 4, figure=fig, wspace=0.08, hspace=0.35)

CLASS_DATA   = [('Alpha Helix', X_alpha, '#2980b9'),
                ('Beta Strand', X_beta,  '#27ae60'),
                ('Random Coil', X_coil,  '#8e44ad')]
N_EXAMPLES   = 3
CMAP_DISPLAY = 'viridis_r'

for row, (class_name, X_class, color) in enumerate(CLASS_DATA):
    # Class label on left
    label_ax = fig.add_subplot(gs[row, 0])
    label_ax.axis('off')
    label_ax.text(0.5, 0.5, class_name, ha='center', va='center',
                  fontsize=12, fontweight='bold', color=color,
                  transform=label_ax.transAxes, rotation=0)

    for col in range(N_EXAMPLES):
        ax = fig.add_subplot(gs[row, col + 1])
        cmap_img = X_class[col].reshape(N_RESIDUES, N_RESIDUES)
        im = ax.imshow(cmap_img, cmap=CMAP_DISPLAY,
                       vmin=0, vmax=THRESHOLD_ANG * 1.5)
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f'Example {col+1}', fontsize=9, color='#555')

fig.suptitle(
    f'Contact Map Gallery ({N_RESIDUES}-residue peptides, threshold={THRESHOLD_ANG} Å)\n'
    'Colour = Cα–Cα distance (darker = closer)',
    fontsize=12, fontweight='bold', y=1.01
)

# Shared colourbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap=CMAP_DISPLAY,
                            norm=plt.Normalize(0, THRESHOLD_ANG * 1.5))
fig.colorbar(sm, cax=cbar_ax, label='Cα–Cα distance (Å)')

plt.show()

print('Observation: Each fold class has a distinct "fingerprint":')
print('  Alpha: strong off-diagonal bands at |i-j|=3,4 (helical H-bond geometry)')
print('  Beta:  sparse, extended — few contacts beyond nearest neighbours')
print('  Coil:  noisy, irregular — no persistent pattern')


## 📊 Step 4: Mean Contact Maps — The Average Fingerprint

Averaging across the full dataset reveals the **class prototype** — the
expected contact pattern for each fold. This is what the classifier is learning:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

for ax, (class_name, X_class, color) in zip(axes, CLASS_DATA):
    mean_map = X_class.mean(axis=0).reshape(N_RESIDUES, N_RESIDUES)
    im = ax.imshow(mean_map, cmap='viridis_r', vmin=0, vmax=THRESHOLD_ANG * 1.2)
    ax.set_title(f'Mean: {class_name}\n(n={len(X_class):,})',
                 fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Residue j'); ax.set_ylabel('Residue i')
    fig.colorbar(im, ax=ax, shrink=0.8, label='Avg Cα dist (Å)')

fig.suptitle('Class-Prototype Contact Maps — averaged over full training set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 🤖 Step 5: Train a Fold Classifier

We flatten each contact map into a 225-dimensional feature vector and train
a **logistic regression** — deliberately choosing the simplest possible model.
If the patterns are real, even a linear classifier should achieve high accuracy.

> *If a linear model can't learn it, the signal isn't there.
> If it can, neural networks will only do better.*


In [ ]:
# Train/test split — stratified to balance classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} structures')
print(f'Test set:     {X_test.shape[0]} structures')

# Logistic regression — L2 regularised, multinomial
clf = LogisticRegression(max_iter=1000, C=0.1, solver='lbfgs',
                         random_state=42)

t_train = time.time()
clf.fit(X_train, y_train)
print(f'Training time: {time.time()-t_train:.3f}s')

accuracy = clf.score(X_test, y_test)
print(f'Test accuracy: {accuracy:.1%}')
print()
print(classification_report(y_test, clf.predict(X_test),
                             target_names=['alpha', 'beta', 'coil']))


## 📈 Step 6: Confusion Matrix & Decision Weights

The confusion matrix shows *which* classes are confused. The weight map shows
*which contact pairs* the classifier actually uses — connecting the ML model
back to structural biology:


In [ ]:
le = LabelEncoder().fit(['alpha', 'beta', 'coil'])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))

# ── Confusion matrix ──────────────────────────────────────────────────────
ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test, ax=axes[0],
    display_labels=['alpha', 'beta', 'coil'],
    colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix', fontsize=11, fontweight='bold')

# ── Weight maps per class ─────────────────────────────────────────────────
COLORS = ['#2980b9', '#27ae60', '#8e44ad']
for idx, (class_name, color) in enumerate(
    zip(['Alpha weights', 'Beta weights', 'Coil weights'], COLORS)
):
    coef_map = clf.coef_[idx].reshape(N_RESIDUES, N_RESIDUES)
    vmax = np.abs(coef_map).max()
    im = axes[idx+1].imshow(coef_map, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[idx+1].set_title(class_name, fontsize=11, fontweight='bold', color=color)
    axes[idx+1].set_xlabel('Residue j'); axes[idx+1].set_ylabel('Residue i')
    fig.colorbar(im, ax=axes[idx+1], shrink=0.8)

fig.suptitle('Classifier Decision: Which contact pairs drive the prediction?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Blue cells = contact more present in this class (positively weighted)')
print('  Red  cells = contact less present / present in other classes')
print('  Alpha weights concentrate on the |i-j|=3,4 off-diagonal — exactly the')
print('  helical hydrogen-bond distances the classifier learned without being told.')


## 🔬 Step 7: Does More Data Help? Learning Curves

A key question in any ML project: are we data-limited or model-limited?
If accuracy plateaus quickly, the linear model has extracted all available signal.
If it's still rising, more data (faster hardware) would help:


In [ ]:
# Manual learning curve — avoids sklearn FitFailedWarning when accuracy
# saturates at 100%% early (degenerate CV folds with single-class splits).
# We sample increasing fractions of X_train and measure test accuracy directly.
from sklearn.linear_model import LogisticRegression

_min_n = N_CLASSES * 5   # at least 5 samples per class
_min_frac = max(0.02, _min_n / len(X_train))
fractions = np.linspace(_min_frac, 1.0, 12)
train_ns, train_accs, test_accs = [], [], []

for frac in fractions:
    n = max(N_CLASSES, int(frac * len(X_train)))
    # Stratified subsample of training data
    idx = []
    for cls in ["alpha", "beta", "coil"]:
        cls_idx = np.where(y_train == cls)[0]
        n_cls   = max(1, int(frac * len(cls_idx)))
        idx.extend(cls_idx[:n_cls])
    Xs, ys = X_train[idx], y_train[idx]
    clf_lc = LogisticRegression(max_iter=500, C=0.1,
                                solver="lbfgs", random_state=42)
    clf_lc.fit(Xs, ys)
    train_ns.append(len(idx))
    train_accs.append(clf_lc.score(Xs, ys))
    test_accs.append(clf_lc.score(X_test, y_test))

print("Learning curve computed over {} training-size steps.".format(len(fractions)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_ns, train_accs, "o-", color="#2980b9", label="Training accuracy")
ax.plot(train_ns, test_accs,  "o-", color="#e74c3c", label="Test accuracy")
ax.axhline(1.0, color="#7f8c8d", linestyle=":", linewidth=0.8)
ax.set_xlabel("Training set size (structures)", fontsize=10)
ax.set_ylabel("Accuracy", fontsize=10)
ax.set_title("Learning Curve: Does more data improve the classifier?",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

plateau = test_accs[-1] - test_accs[int(len(test_accs) * 0.5)]
print("Accuracy gain (50%% -> 100%% of data): {:+.3f}".format(plateau))
print("< 0.01: model saturated — larger dataset will not help a linear classifier.")
print("> 0.05: more structures (generated on faster hardware) would improve accuracy.")


## 🎓 Key Takeaways

### What you built
A complete ML pipeline: **structure generation → featurisation → classification →
interpretability**, all within a biologically grounded framework.

### Contact maps encode secondary structure unambiguously
The classifier learns the 3,4-offset diagonal band pattern of alpha helices and the
extended low-contact pattern of beta strands purely from distance matrices —
with no sequence information, no atom names, and no prior knowledge of secondary
structure.

### The time-budget pattern scales automatically
On faster hardware, `GENERATION_TIME_BUDGET_SECONDS = 30` will produce a larger
dataset automatically. Increase it to 120 or 300 seconds on a workstation or
when running overnight — no other code changes required.

### What to try next
- Replace `LogisticRegression` with a **2D convolutional neural network** —
  the contact map is a natural image input for a CNN
- Add a **mixed-structure** class using `structure='1-7:alpha,8-15:beta'`
  and observe where the classifier places it in probability space
- Use the classifier to **screen synthetic decoys** from the Hard Decoy Challenge —
  does it find misfolded structures that the energy score misses?
- Connect to the **Defensibility Dashboard**: structures with low classification
  confidence often also fail Ramachandran checks
